# 02 - Embeddings com Deep Learning

Este notebook e autossuficiente. Toda a logica de carregamento, extracao, visualizacao e classificacao fica aqui.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet50_Weights, resnet50

try:
    import umap
except ImportError:
    umap = None

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATASET_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'CelebA-Spoof'
OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'embeddings' / 'train' / 'deep_resnet50.parquet'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
MAX_SAMPLES = 200

for folder in [OUTPUT_PATH.parent, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATASET_ROOT

## Funcoes locais

In [ ]:
SPLIT_FILES = {
    'train': 'train_label.json',
    'validation': 'test_label.json',
    'test': 'test_label.json',
}

def normalize_label(raw_label):
    if isinstance(raw_label, list) and raw_label:
        return int(raw_label[-1])
    return int(raw_label)

def load_manifest(dataset_root, split='train', max_samples=None):
    dataset_root = Path(dataset_root)
    split_file = SPLIT_FILES.get(split, f'{split}.json')
    manifest_path = dataset_root / 'metas' / 'intra_test' / split_file
    if not manifest_path.exists():
        manifest_path = dataset_root / split_file
    if not manifest_path.exists():
        raise FileNotFoundError(
            "Manifesto da CelebA-Spoof nao encontrado. "
            "Execute antes o notebook 01 de preparacao."
        )

    with manifest_path.open('r', encoding='utf-8') as file:
        data = json.load(file)

    rows = []
    for image_path, raw_metadata in data.items():
        if isinstance(raw_metadata, dict):
            label = raw_metadata.get('live_spoof', raw_metadata.get('label', 0))
        else:
            label = raw_metadata
        rows.append({'image_path': image_path, 'label': normalize_label(label)})

    frame = pd.DataFrame(rows)
    if max_samples:
        frame = frame.head(max_samples).copy()
    return frame

class CelebASpoofDataset(Dataset):
    def __init__(self, dataset_root, split='train', image_size=224, max_samples=None):
        self.dataset_root = Path(dataset_root)
        self.samples = load_manifest(self.dataset_root, split=split, max_samples=max_samples)
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        row = self.samples.iloc[index]
        image = Image.open(self.dataset_root / row['image_path']).convert('RGB')
        return {
            'image': self.transform(image),
            'label': int(row['label']),
            'image_path': str(row['image_path']),
        }

def build_resnet50_embedding_model():
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    model.fc = nn.Identity()
    model.eval()
    return model

def extract_deep_embeddings(dataset_root, split, output_path, batch_size=32, num_workers=0, max_samples=None, device=None):
    dataset = CelebASpoofDataset(dataset_root=dataset_root, split=split, max_samples=max_samples)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    resolved_device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_resnet50_embedding_model().to(resolved_device)

    rows = []
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(resolved_device)
            embeddings = model(images).cpu().numpy()
            labels = batch['label'].tolist()
            image_paths = batch['image_path']
            for embedding, label, image_path in zip(embeddings, labels, image_paths):
                row = {'image_path': image_path, 'label': label}
                for index, value in enumerate(embedding):
                    row[f'f_{index:04d}'] = float(value)
                rows.append(row)

    frame = pd.DataFrame(rows)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(output_path, index=False)
    return frame

def reduce_embeddings(frame, method='pca', random_state=42):
    meta = frame[['image_path', 'label']].copy()
    features = frame.drop(columns=['image_path', 'label'])

    if method == 'pca':
        reducer = PCA(n_components=2, random_state=random_state)
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=random_state, init='pca')
    elif method == 'umap':
        if umap is None:
            raise ImportError("Instale 'umap-learn' para usar UMAP.")
        reducer = umap.UMAP(n_components=2, random_state=random_state)
    else:
        raise ValueError(f'Metodo desconhecido: {method}')

    reduced = reducer.fit_transform(features)
    result = meta.copy()
    result['x'] = reduced[:, 0]
    result['y'] = reduced[:, 1]
    result['method'] = method
    return result

def save_scatter_plot(frame_2d, output_path, title):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=frame_2d, x='x', y='y', hue='label', alpha=0.7, palette='Set2', s=35)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.close()
    return output_path

def train_classical_model(frame, model_name='svm', test_size=0.2, random_state=42):
    x_data = frame.drop(columns=['image_path', 'label'])
    y_data = frame['label']
    x_train, x_test, y_train, y_test = train_test_split(
        x_data,
        y_data,
        test_size=test_size,
        random_state=random_state,
        stratify=y_data,
    )

    if model_name == 'svm':
        estimator = SVC(kernel='rbf', class_weight='balanced')
    elif model_name == 'knn':
        estimator = KNeighborsClassifier(n_neighbors=5)
    elif model_name == 'logreg':
        estimator = LogisticRegression(max_iter=1000, class_weight='balanced')
    else:
        raise ValueError(f'Modelo desconhecido: {model_name}')

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', estimator),
    ])
    pipeline.fit(x_train, y_train)
    predictions = pipeline.predict(x_test)
    return {
        'model_name': model_name,
        'accuracy': accuracy_score(y_test, predictions),
        'macro_f1': f1_score(y_test, predictions, average='macro'),
        'report': classification_report(y_test, predictions),
    }

## Extracao dos embeddings profundos

In [ ]:
deep_df = extract_deep_embeddings(
    dataset_root=DATASET_ROOT,
    split='train',
    output_path=OUTPUT_PATH,
    max_samples=MAX_SAMPLES,
)
deep_df.head()

In [ ]:
deep_df.shape

## Visualizacao

In [ ]:
deep_2d = reduce_embeddings(deep_df, method='pca')
deep_2d.head()

In [ ]:
figure_path = FIGURES_DIR / 'deep_resnet50_pca.png'
save_scatter_plot(deep_2d, figure_path, 'Deep Embeddings - PCA')
figure_path

## Metodo classico sobre embeddings profundos

In [ ]:
result = train_classical_model(deep_df, model_name='svm')
print('Modelo:', result['model_name'])
print('Acuracia:', round(result['accuracy'], 4))
print('Macro F1:', round(result['macro_f1'], 4))
print(result['report'])